[Home](../../README.md)

### Data Wrangling

This is a demonstration of data wrangling using [Pandas](https://pandas.pydata.org/) the library for data analysis and manipulation.

This Jupyter Notepad demonstrates different processes you can apply to your data to prepare it for feature engineering and model training. For this demonstration we will wrangle the diabetes data set you previewed in the last Jupyter Notebook.

> [!Note]
> None of these processes are destructive to the source CSV as long as you save the modified data to a new CSV.

#### Load the required dependencies

In [54]:
# Import frameworks
import pandas as pd

####  Store the data as a local variable

The data frame is a Pandas object that structures your tabular data into an appropriate format. It loads the complete data in memory so it is now ready for preprocessing.

In [55]:
results = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_results.csv")
results = results.drop(columns=["round_type_id", "id", "format_id", "regional_single_record", "regional_average_record"])
results = results[results["event_id"] == "333"]
results = results[results["average"] > 0]

results_all_events = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_results.csv")
results_all_events = results_all_events.drop(columns=["round_type_id", "id", "format_id", "regional_single_record", "regional_average_record", "pos", "best", "average", "competition_id", "person_name", "person_country_id"])

persons = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_persons.csv")
persons = persons.drop(columns=["sub_id"])

competitions = pd.read_csv("/workspaces/2026SE_MLOPSTask_Michael.L/wca_export/csv/WCA_export_competitions.csv")
competitions = competitions.drop(
    columns=[
        "information",
        "external_website",
        "venue",
        "venue_address",
        "venue_details",
        "cell_name",
        "event_specs",
        "delegates",
        "organizers",
        "latitude_microdegrees",
        "longitude_microdegrees",
    ]
)

KeyboardInterrupt: 

#### Dealing with null values

Null values during data analysis can cause runtime errors and unexpected results. It is important to identify null values and deal with them appropriately before training a model.

The `isnull().sum()` method call returns the null values in any column.

In [ ]:
results.isnull().sum()

pos                  0
best                 0
average              0
competition_id       0
event_id             0
person_name          0
person_id            0
person_country_id    0
dtype: int64

In [ ]:
persons.isnull().sum()

name            0
gender        398
wca_id          0
country_id      0
dtype: int64

In [ ]:
competitions.isnull().sum()

id            0
name          0
city_name     0
country_id    0
cancelled     0
year          0
month         0
day           0
end_year      0
end_month     0
end_day       0
dtype: int64

If you have null data there are many ways to deal with the empty/null values. These are the two most common approaches.
1. Remove any row with a null value with a `dropna()` method call.
2. Replace missing values with another value with a `fillna()` method call. Generally, we use mean value for numerical columns because it may cause minimal changes in your mathematical analysis while maintaining the original size of the data.

Students should reflect why this example removes the null 'SEX' but replacing the mean 'Target'?

In [ ]:
# Remove Null values
persons = persons.dropna(subset=['gender'])
persons.isnull().sum()

name          0
gender        0
wca_id        0
country_id    0
dtype: int64

In [ ]:
# Replace Null values with the mean value for the column
# data_frame['Target'] = data_frame['Target'].fillna(data_frame['Target'].mean())
# data_frame.isnull().sum()

#### Remove Duplicates

The `drop_duplicates()` method call can be then stored back onto the data_frame variable removing the duplicates.

In [ ]:
results = results.drop_duplicates(subset=["average"])
results.head()

,pos,best,average,competition_id,event_id,person_name,person_id,person_country_id
0,15,1968,2128,LyonOpen2007,333,Etienne Amany,2007AMAN01,Cote d_Ivoire
1,16,1731,2140,LyonOpen2007,333,Thomas Rouault,2004ROUA01,France
2,17,2305,2637,LyonOpen2007,333,Antoine Simon-Chautemps,2005SIMO01,France
4,19,2677,2906,LyonOpen2007,333,Marlène Desmaisons,2007DESM01,France
5,20,1869,2910,LyonOpen2007,333,Ton Dennenbroek,2003DENN01,Netherlands


In [ ]:
persons.head()

,name,gender,wca_id,country_id
0,Jozsef Borsos,m,1982BORS01,Serbia
1,Roland Brinkmann,m,1982BRIN01,Germany
2,Julian Chilvers,m,1982CHIL01,United Kingdom
3,Jessica Fridrich,f,1982FRID01,USA
4,Jessica Fridrich,f,1982FRID01,Czech Republic


In [ ]:
competitions.head()

,id,name,city_name,country_id,cancelled,year,month,day,end_year,end_month,end_day
0,100Merito2018,100º Mérito 2018,"Santarém, Pará",Brazil,0,2018,4,14,2018,4,14
1,100YearsRepublicAnkara2023,100 Years Republic Ankara 2023,Ankara,Turkey,0,2023,10,28,2023,10,29
2,100YearsRepublicIstanbul2023,100 Years Republic İstanbul 2023,İstanbul,Turkey,0,2023,10,28,2023,10,29
3,100YilMBACubeWeekend2023,100. Yıl MBA Cube Weekend 2023,İstanbul,Turkey,0,2023,12,16,2023,12,17
4,10AniversarioGuatemala2023,Décimo Aniversario Guatemala 2023,Guatemala City,Guatemala,1,2023,10,14,2023,10,15


#### Replace data

We can run a lambda function on a column to modify its values. For a simple example, let’s convert the Sex to lowercase. To run a function over a complete column, we can use the apply method which iterates over each row and modifies the values.

In [ ]:
# data_frame['SEX'] = data_frame['SEX'].apply(lambda x: x.lower())
# data_frame['SEX'].head()

We can check that there are no data entry errors by the `unique()` method call.

In [ ]:
results['event_id'].unique()

<StringArray>
['333']
Length: 1, dtype: str

In [ ]:
# data_frame['SEX'] = data_frame['SEX'].apply(lambda gender: 'male' if gender.lower() == 'male' else 'female')
# data_frame['SEX'].unique()

#### Remove outliers

Outliers can skew your analysis on numerical columns, and it is important to remove them. We can use the 25th and 75th quartile on numerical data, to get the inter-quartile range. This allows us to estimate an acceptable range, and we can then filter out any values outside this range. Mathematically, outliers are values occurring outside 1.5 times the interquartile range (IQR) from the first quartile (Q1) or third quartile (Q3).

In [ ]:
#get the inter-quartile range on the blood pressure column
print(results['average'].describe())
Q1 = results['average'].quantile(0.25)
Q3 = results['average'].quantile(0.75)
IQR = Q3 - Q1
print(f'Outliers are a BP above {Q3 + IQR * 1.5} or below {Q1 - IQR * 1.5}')


count    15288.000000
mean      8717.795199
std       5796.756158
min        384.000000
25%       4230.750000
50%       8052.500000
75%      12030.250000
max      52507.000000
Name: average, dtype: float64
Outliers are a BP above 23729.5 or below -7468.5


In [ ]:
# Filter blood pressure within the acceptable range
results = results[(results['average'] >= Q1 - 1.5 * IQR) & (results['average'] <= Q3 + 1.5 * IQR)]
print(results['average'].describe())
results.head()

count    15015.000000
mean      8342.874259
std       5080.664344
min        384.000000
25%       4162.500000
50%       7916.000000
75%      11793.500000
max      23729.000000
Name: average, dtype: float64


,pos,best,average,competition_id,event_id,person_name,person_id,person_country_id
0,15,1968,2128,LyonOpen2007,333,Etienne Amany,2007AMAN01,Cote d_Ivoire
1,16,1731,2140,LyonOpen2007,333,Thomas Rouault,2004ROUA01,France
2,17,2305,2637,LyonOpen2007,333,Antoine Simon-Chautemps,2005SIMO01,France
4,19,2677,2906,LyonOpen2007,333,Marlène Desmaisons,2007DESM01,France
5,20,1869,2910,LyonOpen2007,333,Ton Dennenbroek,2003DENN01,Netherlands


#### Scaling features to a common range

Scaling the features makes it easier for machine learning algorithms to find the optimal solution, as the different scales of the features do not influence them.

In [ ]:
scale_feature = 'average'

#the minimum value with space for outliers
MIN_AVG = 200

#the maximum value with space for outliers
MAX_AVG = 23750

#scale features
results[scale_feature] = [(X - MIN_AVG) / (MAX_AVG - MIN_AVG) for X in results[scale_feature]]

results.describe()

,pos,best,average
count,15015.000000,15015.000000,15015.000000
mean,57.294572,6494.918415,0.345770
std,58.372608,3789.183806,0.215739
min,1.000000,324.000000,0.007813
25%,22.000000,3410.500000,0.168259
50%,43.000000,6250.000000,0.327643
75%,75.000000,9065.000000,0.492293
max,908.000000,21561.000000,0.999108


> [!important]
> You need to save the calculations for each dataset you scale for scaling new values for prediction. Use [2.1.2.data.records.md](2.1.2.data.records.md) to record this information.

#### Save the wrangled data to CSV

In [ ]:
results.to_csv('../2.2.Feature_Engineering/2.2.1.wrangled_results.csv', index=False)
persons.to_csv("../2.2.Feature_Engineering/2.2.1.wrangled_persons.csv", index=False)
competitions.to_csv("../2.2.Feature_Engineering/2.2.1.wrangled_competitions.csv", index=False)
results_all_events.to_csv("../2.2.Feature_Engineering/2.2.1.unwrangled_results.csv", index=False)